In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Figure 1.3 生成脚本: Limitations of Voxel-wise Pipeline
-------------------------------------------------------
包含两部分内容：
1. Grid Mismatch: 高分辨解剖图 vs 低分辨采集网格 (模拟)
2. Partial Volume: 混合体素的光谱特征 vs 纯净组织光谱

作者: Gemini (Assisting User)
基于: German et al. (NeuroImage 2021) 概念
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from scipy.ndimage import zoom
import h5py
import os

# ================= 配置区域 (请根据实际情况修改) =================

# 1. 数据路径
DATA_PATH = "/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal/ODP_01_qhlazec_3d_validated_minimal.mat"

# 2. 切片选择 (Axial Dim0)
# 建议选一个脑室或者是皮层沟回清晰的层面
SLICE_IDX = 222 

# 3. 放大区域中心 (Crop Center)
# 这是一个关键参数。您需要找到一个 灰质/白质 交界处。
# 如果设置为 None，代码会尝试自动找一个边缘，但手动指定效果最好。
# 格式: (Row/Dim1, Col/Dim2)
CROP_CENTER = None  # e.g., (120, 140)

# 4. 放大窗口大小 (像素)
CROP_SIZE = 40 

# 5. 模拟分辨率比例 (MPRAGE vs CEST)
# MPRAGE ~0.65mm, CEST ~1.8mm -> ratio approx 3
GRID_RATIO = 3 

# 6. 特征索引 (0-based, based on your description)
# MPRAGE: 342 -> 341
IDX_MPRAGE = 341
# Contrast for plotting signature (e.g., uFA or NOE)
IDX_SIG_START = 0
IDX_SIG_END = 341

OUTPUT_FILENAME = "Figure_1_3_Limitations.png"

# ================= 辅助函数 =================

def load_data(mat_path):
    print(f"Loading: {mat_path} ...")
    with h5py.File(mat_path, 'r') as f:
        # 假设数据格式为 [Slices, H, W, Features] 或 [Features, Slices, H, W]
        # 根据您之前的代码，似乎需要转置
        data = f['data'][:]
        if data.shape[0] in [341, 351]: # 如果特征在第一维
            data = np.moveaxis(data, 0, -1)
        region_mask = f['region_mask'][:].astype(bool)
    return data, region_mask

def normalize(arr):
    """鲁棒归一化到 0-1"""
    arr = arr.astype(np.float32)
    p2, p98 = np.percentile(arr[arr > 0], [2, 98])
    return np.clip((arr - p2) / (p98 - p2 + 1e-8), 0, 1)

def find_edge_region(img_slice):
    """自动寻找梯度最大的区域（边缘）作为默认 Crop 中心"""
    grad = np.gradient(img_slice)
    grad_mag = np.sqrt(grad[0]**2 + grad[1]**2)
    # 忽略图像边缘
    margin = 50
    center_map = grad_mag[margin:-margin, margin:-margin]
    ind = np.unravel_index(np.argmax(center_map, axis=None), center_map.shape)
    return (ind[0] + margin, ind[1] + margin)

# ================= 绘图逻辑 =================

def plot_limitations(data, slice_idx, center, crop_size, output_file):
    
    # --- 1. 准备数据 ---
    # 获取背景图像 (MPRAGE)
    img_full = data[slice_idx, :, :, IDX_MPRAGE]
    
    # 确定裁剪区域
    if center is None:
        center = find_edge_region(img_full)
        print(f"Auto-selected crop center: {center}")
    
    r_start = max(0, center[0] - crop_size // 2)
    r_end = min(img_full.shape[0], center[0] + crop_size // 2)
    c_start = max(0, center[1] - crop_size // 2)
    c_end = min(img_full.shape[1], center[1] + crop_size // 2)
    
    img_crop = img_full[r_start:r_end, c_start:c_end]
    img_crop_norm = normalize(img_crop)

    # --- 2. 模拟混合体素 (Partial Volume Voxel) ---
    # 我们选择 Crop 中心的一个 3x3 区域作为 "CEST Voxel"
    # 在这个区域内，我们分别取左上角(Tissue A) 和 右下角(Tissue B) 的特征
    
    # 相对坐标
    cx, cy = crop_size // 2, crop_size // 2
    offset = GRID_RATIO // 2
    
    # 提取三个位置的完整特征向量 (Spectrum)
    # A: Pure Tissue 1 (假设在左上)
    # B: Pure Tissue 2 (假设在右下)
    # M: Mixed Voxel (中心)
    
    # 注意：为了让图好看，我们在局部搜索一下最大差异点
    # 简单起见，取中心和对角线
    vec_mixed = data[slice_idx, r_start+cx, c_start+cy, :341]
    
    # 寻找局部差异最大的两个点作为 Pure A 和 Pure B
    local_patch = data[slice_idx, r_start+cx-2:r_start+cx+3, c_start+cy-2:c_start+cy+3, IDX_MPRAGE]
    p_min_idx = np.unravel_index(np.argmin(local_patch), local_patch.shape)
    p_max_idx = np.unravel_index(np.argmax(local_patch), local_patch.shape)
    
    # 还原到全局坐标提取特征
    # -2 是因为 local_patch 是从 center-2 开始的
    idx_a_global = (r_start + cx - 2 + p_min_idx[0], c_start + cy - 2 + p_min_idx[1])
    idx_b_global = (r_start + cx - 2 + p_max_idx[0], c_start + cy - 2 + p_max_idx[1])
    
    vec_a = data[slice_idx, idx_a_global[0], idx_a_global[1], :341] # Darker tissue (e.g., GM or CSF)
    vec_b = data[slice_idx, idx_b_global[0], idx_b_global[1], :341] # Brighter tissue (e.g., WM)
    
    # 归一化特征向量以便绘图比较
    def norm_vec(v): return (v - np.mean(v)) / (np.std(v) + 1e-8)
    
    sig_mixed = norm_vec(vec_mixed)
    sig_a = norm_vec(vec_a)
    sig_b = norm_vec(vec_b)

    # --- 3. 开始绘图 ---
    fig = plt.figure(figsize=(14, 8), facecolor='white')
    gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1.2], wspace=0.2)
    
    # === Panel A: Spatial Mismatch ===
    ax_img = fig.add_subplot(gs[0])
    
    # 显示 MPRAGE
    ax_img.imshow(img_crop_norm, cmap='gray', origin='upper', extent=[0, crop_size, crop_size, 0])
    
    # 绘制模拟的低分辨率网格 (Coarse Grid)
    # 假设每 GRID_RATIO 个像素画一条线
    for x in range(0, crop_size, GRID_RATIO):
        ax_img.axvline(x, color='cyan', linewidth=0.8, alpha=0.6)
    for y in range(0, crop_size, GRID_RATIO):
        ax_img.axhline(y, color='cyan', linewidth=0.8, alpha=0.6)
        
    # 高亮选中的“混合体素”
    # 网格对齐逻辑：找到包含中心的那个 Grid
    grid_x = (cx // GRID_RATIO) * GRID_RATIO
    grid_y = (cy // GRID_RATIO) * GRID_RATIO
    
    rect_mixed = patches.Rectangle((grid_x, grid_y), GRID_RATIO, GRID_RATIO, 
                                   linewidth=2.5, edgecolor='orange', facecolor='none', zorder=10)
    ax_img.add_patch(rect_mixed)
    
    # 标注 A 和 B 的位置
    # p_min_idx 是相对 5x5 patch 的，需要转换到 crop 坐标
    # Patch start relative to crop: cx - 2
    pos_a_crop = (cx - 2 + p_min_idx[1], cx - 2 + p_min_idx[0]) # x, y for scatter
    pos_b_crop = (cx - 2 + p_max_idx[1], cx - 2 + p_max_idx[0])
    
    ax_img.scatter(pos_a_crop[0], pos_a_crop[1], c='blue', s=30, label='Pure Tissue A', zorder=11, edgecolors='white')
    ax_img.scatter(pos_b_crop[0], pos_b_crop[1], c='green', s=30, label='Pure Tissue B', zorder=11, edgecolors='white')
    
    # 文字标注
    ax_img.text(grid_x, grid_y-1, "Low-Res Voxel\n(CEST/QTI)", color='orange', fontsize=10, fontweight='bold')
    ax_img.set_title("(A) Resolution Mismatch & Interpolation", fontsize=12, fontweight='bold', pad=15)
    ax_img.set_xlabel("High-Res MPRAGE Grid (0.65mm)", fontsize=10)
    
    # 去除刻度
    ax_img.set_xticks([])
    ax_img.set_yticks([])

    # === Panel B: Spectral Mixing ===
    ax_sig = fig.add_subplot(gs[1])
    
    # 为了清晰，只画前 100 个特征 (QTI + Linear b-tensor) 或者选几个有代表性的区域
    # 这里我们画全谱，但平滑一下或者透明度处理
    x_axis = np.arange(341)
    
    # 绘制曲线
    ax_sig.plot(x_axis, sig_a, color='blue', alpha=0.4, linewidth=1, label='Signature: Pure Tissue A')
    ax_sig.plot(x_axis, sig_b, color='green', alpha=0.4, linewidth=1, label='Signature: Pure Tissue B')
    ax_sig.plot(x_axis, sig_mixed, color='orange', alpha=1.0, linewidth=1.5, label='Signature: Mixed Voxel')
    
    # 高亮差异区域 (Highlight discrepancy)
    # 找到混合曲线明显处于两者之间的地方画个圈或箭头
    ax_sig.set_title("(B) The Partial Volume Problem", fontsize=12, fontweight='bold', pad=15)
    ax_sig.set_ylabel("Normalized Intensity (Z-score)", fontsize=10)
    ax_sig.set_xlabel("Feature Index (0-340)", fontsize=10)
    
    # 底部标注模态区域 (像图1.1那样)
    # 0-14 QTI, 15-224 b-tensor, 225-228 CEST, 229-341 Z-spec
    trans = ax_sig.get_xaxis_transform()
    ax_sig.text(7, -0.15, "QTI", transform=trans, ha='center', fontsize=9)
    ax_sig.text(120, -0.15, "Diffusion (b-tensors)", transform=trans, ha='center', fontsize=9)
    ax_sig.text(280, -0.15, "CEST Z-spectra", transform=trans, ha='center', fontsize=9)
    
    # 分割线
    for line in [15, 225]:
        ax_sig.axvline(line, color='gray', linestyle='--', alpha=0.3)

    ax_sig.legend(loc='upper right', fontsize=9)
    
    # 添加一个 "Hard Label" 的讽刺性标注
    # 在混合曲线旁边画个箭头指向一个伪造的标签框
    bbox_props = dict(boxstyle="darrow,pad=0.3", fc="red", ec="red", alpha=0.2)
    ax_sig.text(170, 2, "Current Pipeline:\nForces 'Hard Label'\n(Overconfident)", 
                ha="center", va="center", rotation=0, size=10,
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="red", lw=2))

    plt.tight_layout()
    plt.savefig(output_file, dpi=300)
    print(f"Generated Figure 1.3: {output_file}")
    plt.close()

def main():
    # 1. 加载数据
    data, mask = load_data(DATA_PATH)
    
    # 2. 验证维度
    print(f"Data shape: {data.shape}")
    
    # 3. 生成图片
    plot_limitations(data, SLICE_IDX, CROP_CENTER, CROP_SIZE, OUTPUT_FILENAME)

if __name__ == "__main__":
    main()